# Chapter 7 gallery — robust logistic regression

Reproduces `leukemia.R` (Ex 7.1) and `skin.R` (Ex 7.2) via the `by_logreg` / `wby_logreg` / `wml_logreg` family.

In [ ]:
import os, sys, pathlib

# Windows R_HOME setup (skip if already configured)
if sys.platform == "win32" and "R_HOME" not in os.environ:
    os.environ["R_HOME"] = r"C:\Program Files\R\R-4.5.2"
    os.environ["PATH"] = r"C:\Program Files\R\R-4.5.2\bin\x64;" + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe for CI execution
import matplotlib.pyplot as plt
import robstatm_py as rpm
from robstatm_py import set_seed
from robstatm_py._r import r as _r

ro = _r()
ro.r("suppressMessages(library(RobStatTM))")
FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print(f"robstatm_py {rpm.__version__}")

## leukemia — weighted Bianco–Yohai logistic regression (Example 7.1)

`leukemia.R` fits `logregWBY` (weighted BY M-estimator) to the leukemia survival data and compares deviance residuals with the ML fit. (The `robust::glmRob` cubif comparator is out of scope and omitted.)

In [ ]:
leuk = rpm.datasets.leuk_dat()
print('columns:', list(leuk.columns))
Xl = leuk.iloc[:, :2].to_numpy(dtype=float)
yl = leuk.iloc[:, -1].to_numpy(dtype=float)
wby = rpm.wby_logreg(Xl, yl, intercept=True)
print(wby)
print('coefficients:', np.round(wby.coefficients, 4))
print('std deviation:', np.round(wby.standard_deviation, 4))

### Strict-tier cross-check vs direct R `logregWBY`

In [ ]:
ro.globalenv['Xl'] = Xl
ro.globalenv['yl'] = yl.reshape(-1, 1)
ro.r('rw <- logregWBY(Xl, yl, intercept=1)')
r_coef = np.asarray(ro.r('as.numeric(rw$coefficients)'), dtype=float)
print('coefficients bit-equal to R:', np.array_equal(wby.coefficients, r_coef))

## skin — robust logistic regression family (Example 7.2)

`skin.R` fits the weighted-M (`logregWBY`), plain BY (`logregBY`) and weighted-ML (`logregWML`) estimators to the vaso-constriction data. We reproduce all three; the ML and cubif comparators are out of scope.

In [ ]:
skin = rpm.datasets.skin()
print('columns:', list(skin.columns))
Xs = skin.iloc[:, :2].to_numpy(dtype=float)
ys = skin['vasoconst'].to_numpy(dtype=float)
wby = rpm.wby_logreg(Xs, ys, intercept=True)
by = rpm.by_logreg(Xs, ys, intercept=True)
wml = rpm.wml_logreg(Xs, ys, intercept=True)
for name, fit in (('WBY', wby), ('BY', by), ('WML', wml)):
    print(f'{name:>3}: coef = {np.round(fit.coefficients, 4)}')

In [ ]:
# Figure 7.5 analogue: sorted |deviance residuals| of the weighted-M fit
dev = np.sort(np.abs(wby.residual_deviances))
pp = (np.arange(1, len(dev)+1) - 0.5) / len(dev)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(pp, dev, 'o-', ms=4)
ax.set_xlabel('quantiles'); ax.set_ylabel('|deviance residuals|')
ax.set_title('skin — weighted-M deviance residuals (Fig 7.5 analogue)')
fig.savefig(FIG_DIR / 'ch7_skin.png', dpi=110, bbox_inches='tight'); plt.close(fig)
print('done')